# Data Pipeline Walkthrough

**Pattern: Load → Clean → Combine → Train → Infer**

This notebook demonstrates each dataset being loaded and precleaned independently,
then combined into the ML-ready training dataset.

### Data Sources (from `data/input/platinum/`)

| # | Dataset | Size | Purpose |
|---|---------|------|---------|
| 1 | `site_scores_revenue_and_diagnostics.csv` | 927 MB | Monthly revenue & metrics per site (primary) |
| 2 | `nearest_site_distances.csv` | 6.5 MB | Pre-computed nearest-neighbor site distances |
| 3 | `site_interstate_distances.csv` | 3.8 MB | Site-to-interstate distances (multiple per site) |
| 4 | `mcdonalds_geodata.csv` | 7.5 MB | McDonald's store locations (raw lat/lon) |
| 5 | `walmart_geodata.csv` | 6.7 MB | Walmart store locations (raw lat/lon) |
| 6 | `target_geo_data.csv` | 1.6 MB | Target store locations (raw lat/lon) |

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on the path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import polars as pl
import numpy as np

DATA_DIR = PROJECT_ROOT / "data" / "input" / "platinum"
print(f"Data directory: {DATA_DIR}")
print(f"Files:")
for f in sorted(DATA_DIR.iterdir()):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:50s} {size_mb:>8.1f} MB")

---
## 1. Load: Site Scores (Primary Dataset)

The main dataset — **monthly** revenue and diagnostic metrics for every site.
Each row is one site × one month. Key columns:

- `id_gbase`: Primary site identifier (used for grouping)
- `gtvid`: Geospatial identifier (used for distance joins)
- `date`: Monthly timestamp
- `revenue`, `monthly_impressions`, `monthly_nvis`: Performance metrics
- `statuis`: Site status (**note the typo** — this is `status` misspelled in the source data)
- 30+ restriction flags (`r_*`), 9 capability flags (`c_*`)

In [ ]:
# Load with explicit null handling — source uses multiple null representations
site_scores = pl.read_csv(
    DATA_DIR / "site_scores_revenue_and_diagnostics.csv",
    infer_schema_length=10000,
    null_values=["", "NA", "null", "Unknown"],
)

print(f"Shape: {site_scores.shape[0]:,} rows × {site_scores.shape[1]} columns")
print(f"Unique sites (id_gbase): {site_scores['id_gbase'].n_unique():,}")
print(f"Date range: {site_scores['date'].min()} → {site_scores['date'].max()}")
print(f"\nColumn types:")
for dtype in site_scores.dtypes:
    count = sum(1 for d in site_scores.dtypes if d == dtype)
print(f"  {dict(zip([str(d) for d in set(site_scores.dtypes)], [sum(1 for d2 in site_scores.dtypes if d2 == d) for d in set(site_scores.dtypes)]))}")
site_scores.head(3)

In [ ]:
# Preclean: inspect key columns for nulls and anomalies
key_cols = ["id_gbase", "gtvid", "revenue", "statuis", "latitude", "longitude"]
for col in key_cols:
    null_count = site_scores[col].null_count()
    null_pct = null_count / len(site_scores) * 100
    print(f"  {col:30s}  nulls: {null_count:>8,} ({null_pct:.1f}%)")

print(f"\nStatus values (statuis column):")
print(site_scores["statuis"].value_counts().sort("count", descending=True))

---
## 2. Load: Nearest Site Distances

Pre-computed distance from each site to its nearest neighboring site.
One row per site. Join key: `GTVID` (uppercase) ↔ `gtvid` (lowercase in main data).

In [ ]:
nearest_sites = pl.read_csv(DATA_DIR / "nearest_site_distances.csv")

print(f"Shape: {nearest_sites.shape[0]:,} rows × {nearest_sites.shape[1]} columns")
print(f"Columns: {nearest_sites.columns}")
print(f"\nDistance stats (miles):")
print(nearest_sites["nearest_site_distance_mi"].describe())
nearest_sites.head(3)

In [ ]:
# Preclean: select only the columns needed for the join
nearest_clean = nearest_sites.select([
    "GTVID",
    "nearest_site",
    pl.col("nearest_site_distance_mi").alias("min_distance_to_nearest_site_mi"),
])

null_count = nearest_clean["min_distance_to_nearest_site_mi"].null_count()
print(f"Precleaned: {len(nearest_clean):,} sites, {null_count} null distances")
nearest_clean.head(3)

---
## 3. Load: Interstate Distances

Distance from each site to nearby interstates. **Multiple rows per site** (one per interstate),
so we aggregate to keep only the minimum distance.

In [ ]:
interstate_raw = pl.read_csv(DATA_DIR / "site_interstate_distances.csv")

print(f"Shape: {interstate_raw.shape[0]:,} rows × {interstate_raw.shape[1]} columns")
print(f"Unique sites: {interstate_raw['GTVID'].n_unique():,}")
print(f"Rows per site: {interstate_raw.shape[0] / interstate_raw['GTVID'].n_unique():.1f} avg")
print(f"Columns: {interstate_raw.columns}")
interstate_raw.head(3)

In [ ]:
# Preclean: aggregate to one row per site (minimum distance)
interstate_clean = interstate_raw.group_by("GTVID").agg([
    pl.col("distance_to_interstate_mi").min().alias("min_distance_to_interstate_mi"),
    pl.col("nearest_interstate").first().alias("nearest_interstate"),
])

print(f"Precleaned: {len(interstate_clean):,} unique sites")
print(f"Distance range: {interstate_clean['min_distance_to_interstate_mi'].min():.2f} – {interstate_clean['min_distance_to_interstate_mi'].max():.2f} mi")
interstate_clean.head(3)

---
## 4. Load: McDonald's Geodata

Raw store locations (lat/lon) — **not** pre-computed distances.
We compute the Haversine distance from each site to the nearest McDonald's.

In [ ]:
mcdonalds_raw = pl.read_csv(DATA_DIR / "mcdonalds_geodata.csv")

print(f"Shape: {mcdonalds_raw.shape[0]:,} stores × {mcdonalds_raw.shape[1]} columns")
print(f"Columns: {mcdonalds_raw.columns}")
null_coords = mcdonalds_raw.filter(
    pl.col("latitude").is_null() | pl.col("longitude").is_null()
)
print(f"Stores missing coordinates: {len(null_coords)}")
mcdonalds_raw.select(["dba", "city", "state", "latitude", "longitude"]).head(3)

In [ ]:
# Preclean: keep only valid coordinates for distance computation
mcdonalds_clean = mcdonalds_raw.filter(
    pl.col("latitude").is_not_null() & pl.col("longitude").is_not_null()
).select(["latitude", "longitude"])

print(f"Precleaned: {len(mcdonalds_clean):,} stores with valid coordinates")
print(f"Lat range: {mcdonalds_clean['latitude'].min():.2f} – {mcdonalds_clean['latitude'].max():.2f}")
print(f"Lon range: {mcdonalds_clean['longitude'].min():.2f} – {mcdonalds_clean['longitude'].max():.2f}")
mcdonalds_clean.head(3)

---
## 5. Load: Walmart Geodata

In [ ]:
walmart_raw = pl.read_csv(DATA_DIR / "walmart_geodata.csv")

print(f"Shape: {walmart_raw.shape[0]:,} stores × {walmart_raw.shape[1]} columns")
null_coords = walmart_raw.filter(
    pl.col("latitude").is_null() | pl.col("longitude").is_null()
)
print(f"Stores missing coordinates: {len(null_coords)}")

In [ ]:
# Preclean: keep only valid coordinates
walmart_clean = walmart_raw.filter(
    pl.col("latitude").is_not_null() & pl.col("longitude").is_not_null()
).select(["latitude", "longitude"])

print(f"Precleaned: {len(walmart_clean):,} stores with valid coordinates")
walmart_clean.head(3)

---
## 6. Load: Target Geodata

In [ ]:
target_raw = pl.read_csv(DATA_DIR / "target_geo_data.csv")

print(f"Shape: {target_raw.shape[0]:,} stores × {target_raw.shape[1]} columns")
null_coords = target_raw.filter(
    pl.col("latitude").is_null() | pl.col("longitude").is_null()
)
print(f"Stores missing coordinates: {len(null_coords)}")

In [ ]:
# Preclean: keep only valid coordinates
target_clean = target_raw.filter(
    pl.col("latitude").is_not_null() & pl.col("longitude").is_not_null()
).select(["latitude", "longitude"])

print(f"Precleaned: {len(target_clean):,} stores with valid coordinates")
target_clean.head(3)

---
## 7. Combine: Aggregate Monthly → Site-Level

Collapse 1.4M monthly rows into one row per site (~60K sites).
This computes totals, averages, and **multi-horizon momentum features** (relative strength).

In [ ]:
from site_scoring.data_transform import aggregate_site_metrics

site_agg = aggregate_site_metrics(site_scores)

print(f"\nAggregated: {len(site_agg):,} sites × {len(site_agg.columns)} columns")
print(f"\nKey derived columns:")
for col in site_agg.columns:
    if col.startswith("rs_") or col.startswith("avg_monthly") or col.startswith("total_"):
        print(f"  {col}")

---
## 8. Combine: Compute Chain Distances (Haversine)

The McDonald's, Walmart, and Target files are raw store locations.
We compute the minimum Haversine distance from each site to the nearest store in each chain.

In [ ]:
from site_scoring.data_transform import compute_chain_distances

# Compute distances from each site to nearest chain store
mcdonalds_distances = compute_chain_distances(site_agg, DATA_DIR / "mcdonalds_geodata.csv", "mcdonalds")
walmart_distances = compute_chain_distances(site_agg, DATA_DIR / "walmart_geodata.csv", "walmart")
target_distances = compute_chain_distances(site_agg, DATA_DIR / "target_geo_data.csv", "target")

print(f"\nComputed distance DataFrames:")
for name, df in [("McDonald's", mcdonalds_distances), ("Walmart", walmart_distances), ("Target", target_distances)]:
    print(f"  {name}: {len(df):,} sites, columns = {df.columns}")

---
## 9. Combine: Join All Geospatial Features

Left-join all distance features onto the site-level DataFrame.
Join key: `gtvid` (lowercase) ↔ `GTVID` (uppercase in pre-computed files) or `gtvid` (chain distances).

In [ ]:
from site_scoring.data_transform import join_geospatial_features

site_combined = join_geospatial_features(
    site_agg,
    nearest_clean,
    interstate_clean,
    mcdonalds_distances,
    walmart_distances,
    target_distances,
)

print(f"\nCombined: {len(site_combined):,} sites × {len(site_combined.columns)} columns")

# Show the distance columns we just joined
dist_cols = [c for c in site_combined.columns if "distance" in c.lower()]
print(f"\nDistance features ({len(dist_cols)}):")
for col in dist_cols:
    matched = site_combined.filter(pl.col(col).is_not_null()).shape[0]
    print(f"  {col:45s}  matched: {matched:,} sites")

---
## 10. Clean: Prepare Training Dataset

Filter to Active sites, remove negative revenue, bin high-cardinality categoricals,
add log transformations, and one-hot encode boolean flags.

In [ ]:
from site_scoring.data_transform import prepare_training_dataset

train_df = prepare_training_dataset(site_combined, active_only=True)

print(f"\nTraining dataset: {len(train_df):,} sites × {len(train_df.columns)} columns")

# Categorize the columns
log_cols = [c for c in train_df.columns if c.startswith("log_")]
encoded_cols = [c for c in train_df.columns if c.endswith("_encoded")]
rs_cols = [c for c in train_df.columns if c.startswith("rs_")]

print(f"\nColumn breakdown:")
print(f"  Log-transformed:  {len(log_cols)}")
print(f"  Encoded booleans: {len(encoded_cols)}")
print(f"  Momentum (RS):    {len(rs_cols)}")
print(f"  Other:            {len(train_df.columns) - len(log_cols) - len(encoded_cols) - len(rs_cols)}")

---
## 11. Train: Feature Processing → Tensors

The `FeatureProcessor` converts the cleaned Polars DataFrame into PyTorch tensors:
- **Numeric**: StandardScaler → clip to [-10, 10]
- **Categorical**: LabelEncoder → integer codes
- **Boolean**: cast to float32
- **Target**: StandardScaler (regression) or binarize (lookalike)

In [ ]:
from site_scoring.config import Config
from site_scoring.data.outputs.tensor import FeatureProcessor

config = Config()
processor = FeatureProcessor(config)
bundle = processor.fit_transform(train_df)

print(f"Tensor shapes:")
print(f"  Numeric:     {bundle.numeric.shape}  (dtype: {bundle.numeric.dtype})")
print(f"  Categorical: {bundle.categorical.shape}  (dtype: {bundle.categorical.dtype})")
print(f"  Boolean:     {bundle.boolean.shape}  (dtype: {bundle.boolean.dtype})")
print(f"  Target:      {bundle.target.shape}  (dtype: {bundle.target.dtype})")

print(f"\nCategorical vocab sizes:")
for name, size in processor.categorical_vocab_sizes.items():
    print(f"  {name}: {size} categories")

In [ ]:
# Verify numeric scaling is well-behaved
print(f"Numeric stats after scaling:")
print(f"  Mean: {bundle.numeric.mean():.4f}  (should be near 0)")
print(f"  Std:  {bundle.numeric.std():.4f}   (should be near 1)")
print(f"  Min:  {bundle.numeric.min():.4f}   (clipped to -10)")
print(f"  Max:  {bundle.numeric.max():.4f}   (clipped to +10)")

print(f"\nTarget stats (scaled):")
print(f"  Mean: {bundle.target.mean():.4f}")
print(f"  Std:  {bundle.target.std():.4f}")

---
## 12. Train: Create DataLoaders & Train Model

Split into train/val/test (70/15/15), create PyTorch DataLoaders,
and run a quick 3-epoch training to verify the full pipeline works end-to-end.

In [ ]:
from site_scoring.data_loader import create_data_loaders
from site_scoring.model import SiteScoringModel
import torch

# Use a fast config for demonstration
demo_config = Config(
    epochs=3,
    batch_size=2048,
    hidden_dims=[128, 64],
    device="mps" if torch.backends.mps.is_available() else "cpu",
)

train_loader, val_loader, test_loader, demo_processor = create_data_loaders(demo_config)

print(f"DataLoaders created:")
print(f"  Train: {len(train_loader.dataset):,} samples, {len(train_loader)} batches")
print(f"  Val:   {len(val_loader.dataset):,} samples")
print(f"  Test:  {len(test_loader.dataset):,} samples")
print(f"  Device: {demo_config.device}")

In [ ]:
# Create model and run quick training
model = SiteScoringModel.from_config(demo_config, demo_processor.categorical_vocab_sizes)
model = model.to(demo_config.device)

optimizer = torch.optim.AdamW(model.parameters(), lr=demo_config.learning_rate)
criterion = torch.nn.HuberLoss()

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nTraining (3 epochs):")

for epoch in range(3):
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    for batch in train_loader:
        numeric = batch[0].to(demo_config.device)
        categorical = batch[1].to(demo_config.device)
        boolean = batch[2].to(demo_config.device)
        target = batch[3].to(demo_config.device)

        optimizer.zero_grad()
        output = model(numeric, categorical, boolean)
        loss = criterion(output.squeeze(), target.squeeze())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1

    avg_loss = epoch_loss / n_batches
    print(f"  Epoch {epoch+1}/3  loss: {avg_loss:.4f}")

print("\nTraining complete.")

---
## 13. Infer: Predict on Test Set

Use the trained model to predict on the held-out test set,
then inverse-transform predictions back to the original revenue scale.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for batch in test_loader:
        numeric = batch[0].to(demo_config.device)
        categorical = batch[1].to(demo_config.device)
        boolean = batch[2].to(demo_config.device)
        target = batch[3]

        output = model(numeric, categorical, boolean)
        all_preds.append(output.cpu().numpy().flatten())
        all_targets.append(target.numpy().flatten())

preds_scaled = np.concatenate(all_preds)
targets_scaled = np.concatenate(all_targets)

# Inverse transform to original revenue scale
preds_original = demo_processor.inverse_transform_target(preds_scaled)
targets_original = demo_processor.inverse_transform_target(targets_scaled)

mae = mean_absolute_error(targets_original, preds_original)
r2 = r2_score(targets_original, preds_original)

print(f"Test Set Results ({len(preds_original):,} sites):")
print(f"  MAE:  ${mae:,.0f}/month")
print(f"  R²:   {r2:.4f}")
print(f"  Pred range:   ${preds_original.min():,.0f} – ${preds_original.max():,.0f}")
print(f"  Actual range: ${targets_original.min():,.0f} – ${targets_original.max():,.0f}")

---
## Pipeline Summary

```
site_scores_revenue_and_diagnostics.csv  (1.4M rows, monthly)
        │
        ▼
  aggregate_site_metrics()  ───────────── (~60K sites, one row each)
        │
        │   ┌── nearest_site_distances.csv  (pre-computed)
        │   ├── site_interstate_distances.csv  (aggregated to min)
        │   ├── mcdonalds_geodata.csv  → Haversine distance
        │   ├── walmart_geodata.csv  → Haversine distance
        │   └── target_geo_data.csv  → Haversine distance
        ▼
  join_geospatial_features()  ─────────── (+5 distance columns)
        │
        ▼
  prepare_training_dataset()  ─────────── (Active only, log transforms, encode flags)
        │                                  → ~26K sites × 111 columns
        ▼
  FeatureProcessor.fit_transform()  ────── (Scale, encode, tensorize)
        │
        ▼
  create_data_loaders()  ──────────────── (70/15/15 split, batched)
        │
        ▼
  SiteScoringModel.forward()  ─────────── (Embeddings + Residual MLP)
        │
        ▼
  inverse_transform_target()  ─────────── (Back to $/month)
```